# Quickstart

Model, dataset, one batch. Needs `data/openfake_train.csv` from `scripts/download_data.py`.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import yaml, torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from src.model import Detector, family_loss, score
from src.data import ImageDataset, load_rows

cfg = yaml.safe_load(open('configs/train.yaml'))

## Как устроена модель

**Идея.** Одна общая модель зрения и по одной маленькой голове на каждое семейство генераторов: OpenAI, Google, Midjourney, Flux, SDXL и так далее, всего 11. Голова отвечает на один вопрос: картинку сделало её семейство или картинка реальная. Получается ансамбль узких детекторов по цене одной модели. Новая версия знакомого генератора обычно похожа на своё семейство, и её ловит его голова.

**1. Модель зрения.** DINOv3 ViT-L/16 от Meta, обучена на 1.7 млрд картинок. Картинка 512×512 режется на 1024 патча 16×16, каждый патч на выходе описан вектором из 1024 чисел. Первые 16 блоков из 24 остаются как есть, последние 8 дообучаются. Целиком замороженная модель с маленькой головой не разделяла даже обучающие данные: после 5 эпох 7 % ложных срабатываний на них. Векторы патчей усредняются в один вектор картинки:

$$f = \frac{1}{N}\sum_{i=1}^{N} f_i \in \mathbb{R}^{1024}$$

**2. Головы.** Один линейный слой, по строке на семейство. $p_k$ — вероятность, что картинку сделало семейство $k$:

$$z_k = w_k^\top f + b_k, \qquad p_k = \sigma(z_k)$$

**3. Чему учим.** Реальная картинка — отрицательный пример для каждой головы. Сгенерированная — положительный пример для головы своего семейства, а остальные головы на ней не учатся: голова Flux не обязана называть картинку GPT реальной. Лосс — бинарная кросс-энтропия, усреднённая по всем таким парам «картинка, голова».

**4. Балл картинки.** Самая уверенная голова, и видно, какая именно:

$$s = \max_k p_k$$

**5. Порог.** Порог $\tau$ подбирается только по реальным картинкам, которых модель не видела на обучении. Он ставится так, чтобы модель ложно срабатывала не больше чем на 1 % из них. Если $s > \tau$, картинка сгенерирована.

$$\tau = s_{(\lceil (n+1)(1-\alpha) \rceil)} \text{ среди баллов } n \text{ отложенных реальных}, \qquad \alpha = 0.01$$

In [ ]:
# model: DINOv3 with its last blocks trained, one head per family
families = list(cfg['model']['families'])
model = Detector(cfg['model']['backbone'], families, cfg['model']['train_blocks'])
trained = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(len(families), 'heads |', f'{trained / 1e6:.0f}M of {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters are trained')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

In [ ]:
# dataset: whole images resized to 512 x 512, with augmentations; label 0 = real, k = family k
number = {g: k for k, name in enumerate(families, 1) for g in cfg['model']['families'][name]}
rows = load_rows('data/openfake_train.csv')
for r in rows:
    r['label'] = number[r['generator']] if r['generator'] else 0
dataset = ImageDataset(rows, cfg['train']['size'], train=True)
image, label = next(iter(DataLoader(dataset, batch_size=4, shuffle=True)))
image.shape, label

In [ ]:
# forward and loss
logits = model(image.to(device))
loss = family_loss(logits, label.to(device))
logits.shape, score(logits), loss.item()

In [ ]:
# the views the model sees
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, im, k in zip(ax, image, label):
    a.imshow(im.permute(1, 2, 0))
    a.set_title('real' if k == 0 else families[k - 1]); a.axis('off')

## Данные

**Обучение: [OpenFake](https://arxiv.org/abs/2509.09495).** Лицензия CC BY-NC 4.0, только для исследований. Реальные фото взяты из LAION и Pexels: публичные люди, новости, мемы с текстом. Для каждого фото нейросеть написала подпись, и по этим подписям картинки сгенерировали 72 модели: GPT Image, Nano Banana, Imagen, Midjourney, DALL·E 3, Flux, SD от 1.x до 3.5, файнтюны с Civitai и кадры видеомоделей. Поэтому реальные фото и фейки похожи по сюжетам, и отличить их по содержанию нельзя.

| Сплит | Строк | Объём | Что внутри |
|---|---|---|---|
| core/train | 2.3M | 3.3 TB | реальные из LAION и Pexels, 72 генератора |
| core/test | 91k | 68 GB | только новые генераторы: GPT Image 1.5 и 2, Nano Banana Pro, Midjourney 7, Flux.2 klein 9B, Z-Image Turbo и другие; реальные из DOCCI и ImageNet |

**Что берём.** Файлы train скачиваются целиком, из каждого берутся нужные картинки, и файл удаляется. Это все 608 файлов: до `n_per_model` картинок на каждый генератор из списка `generators` в конфиге и `n_real` реальных. Генераторы очень неравны: у SD 3.5 около 150k картинок, у Nano Banana 2 около двух тысяч, и у таких берём всё, что попалось. Модель `tiny-random-sana` выкидываем, её картинки сплошной шум. 15 % реальных откладываются для порога, ещё 5000 картинок для валидации. К остальным реальным добавляются отрисованные кодом тексты, интерфейсы, графики и логотипы, 5 % от реальных (`render_train`): так модель реже срабатывает на скриншотах и документах. Цена: сгенерированные логотипы и инфографику она ловит хуже. На тесте у таких картинок другие шрифты.

**Хранение и обработка.** Картинки лежат на диске как есть, байт в байт из датасета: фейки в основном PNG, реальные фото в основном JPEG. Картинки меньше 512 пикселей пропускаются. Картинка, которая лежит не в JPEG, сначала один раз сжимается в JPEG, как будто её уже публиковали: так история сжатия есть у обоих классов. Дальше по порядку, одинаково для обоих классов, как с картинками бывает в интернете: случайный кусок на 50–100 % площади с пропорциями от 1:2 до 2:1, какими бы ни были пропорции самой картинки (так растяжение до квадрата одинаково у обоих классов); в исходном разрешении иногда пересэмплинг, размытие и шум, затем сжатие в JPEG или WebP со случайным качеством от 60 до 95, иногда дважды. Только после этого картинка целиком сжимается до 512×512 без сохранения пропорций, как обычно у моделей зрения, и половина отражается. Все кадры одного размера, поэтому батчи обычные, по 32.

**Батч.** Половина батча реальные фото, а вторую половину семейства генераторов делят поровну: $P(\text{реальное}) = \tfrac12$, $P(\text{семейство } k) = \tfrac{1}{2K}$. Так каждая голова получает столько же примеров своего семейства, сколько остальные.

**Тест.** Картинка целиком сжимается до 512×512, как на обучении, ничего не обрезается.
- **OpenFake test:** доля пойманных фейков от невиденных генераторов при 1 % ложных срабатываний на реальных из DOCCI.
- **SID_Set:** полностью сгенерированные картинки и локальные правки. На правках модель не училась, это честная проверка, ловит ли она их.
- **Отрисованные тексты, интерфейсы, графики и логотипы:** сколько ложных срабатываний на нефото.

In [ ]:
# сколько картинок скачано от каждого генератора
from collections import Counter

rows = load_rows('data/openfake_train.csv')
counts = Counter(r['generator'] or 'real' for r in rows)
print(len(counts) - 1, 'генераторов,', len(rows), 'картинок')
for name, n in counts.most_common():
    print(f'{n:7d}  {name}')

In [ ]:
# 5 случайных картинок одного генератора
import random
from PIL import Image

gen = 'gpt-image-1'  # любое имя из списка выше, или 'real'
pool = [r for r in rows if (r['generator'] or 'real') == gen]
pick = random.sample(pool, min(5, len(pool)))
fig, ax = plt.subplots(1, len(pick), figsize=(4 * len(pick), 4), squeeze=False)
for a, r in zip(ax[0], pick):
    img = Image.open(r['path']).convert('RGB')
    a.imshow(img)
    a.set_title(f'{gen}\n{img.width}×{img.height}', fontsize=9)
    a.axis('off')